# Semana 13: Prompt Engineering y Optimización (LoRA, Cuantización)
## Notebook Conceptual (NB1) – Experimentos con Prompts

**Propósito:** Dominar técnicas de prompting y métodos eficientes para adaptar y optimizar LLMs.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Probar zero-shot, few-shot y chain-of-thought en modelos generativos.
- Comparar respuestas y analizar la mejora con ejemplos.
- Introducir LoRA con PEFT para fine-tuning eficiente.
- Cuantizar un modelo con torch.ao (INT8) y medir reducción de tamaño y velocidad.

---

## 0. Configuración Inicial

Importamos las librerías necesarias e instalamos dependencias.

In [ ]:
# Instalamos dependencias
!pip install transformers torch accelerate peft bitsandbytes

# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import time
import warnings
warnings.filterwarnings('ignore')

# Transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# PEFT para LoRA
from peft import LoraConfig, get_peft_model, TaskType

# Para cuantización
import torch.ao.quantization as tq

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Verificar dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

print("\nLibrerías importadas correctamente.")

---
## 1. Carga del Modelo Base

Usaremos GPT-2 como modelo base para los experimentos de prompting.

In [ ]:
# Cargar modelo y tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Configurar token de padding
tokenizer.pad_token = tokenizer.eos_token

# Crear pipeline de generación
generator = pipeline('text-generation', model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

print("Modelo GPT-2 cargado correctamente.")

---
## 2. Zero-Shot Prompting

El modelo recibe una instrucción directa sin ejemplos previos.

In [ ]:
# Definir prompts zero-shot
zero_shot_prompts = [
    "Traduce 'hello' al español:",
    "Clasifica el sentimiento de esta frase: 'Me encanta este producto' (positivo/negativo):",
    "Responde: ¿Cuál es la capital de Francia?"
]

print("=== ZERO-SHOT PROMPTING ===\n")
for prompt in zero_shot_prompts:
    response = generator(prompt, max_length=50, num_return_sequences=1)[0]['generated_text']
    print(f"Prompt: {prompt}")
    print(f"Respuesta: {response[len(prompt):].strip()}")
    print("-"*50)

---
## 3. Few-Shot Prompting

Proporcionamos ejemplos para guiar al modelo.

In [ ]:
# Few-shot para traducción
few_shot_translation = """
Inglés: cat -> Español: gato
Inglés: dog -> Español: perro
Inglés: house -> Español: casa
Inglés: hello -> Español:"""

# Few-shot para clasificación de sentimiento
few_shot_sentiment = """
Frase: Me encanta esta película -> Sentimiento: positivo
Frase: Es un producto terrible -> Sentimiento: negativo
Frase: Está bien, nada especial -> Sentimiento: neutro
Frase: Me encanta este producto -> Sentimiento:"""

few_shot_prompts = [few_shot_translation, few_shot_sentiment]

print("=== FEW-SHOT PROMPTING ===\n")
for prompt in few_shot_prompts:
    response = generator(prompt, max_length=100, num_return_sequences=1)[0]['generated_text']
    print(f"Prompt:\n{prompt}")
    print(f"\nRespuesta completa:\n{response}")
    print("-"*80)

---
## 4. Chain-of-Thought (CoT) Prompting

Pedimos al modelo que razone paso a paso antes de dar la respuesta.

In [ ]:
# Problema matemático con CoT
cot_prompt = """
P: Si compro 3 manzanas a 50 céntimos cada una y pago con un billete de 5 euros, ¿cuánto cambio recibiré?
Pensémoslo paso a paso:
1. Primero, calculo el coste total de las manzanas: 3 manzanas × 0.50 euros = 1.50 euros.
2. Luego, resto el coste del dinero entregado: 5 euros - 1.50 euros = 3.50 euros.
3. Por lo tanto, el cambio es 3.50 euros.

P: Si tengo 2 docenas de huevos y uso 5 para una tortilla, ¿cuántos me quedan?
Pensémoslo paso a paso:"""

response_cot = generator(cot_prompt, max_length=200, num_return_sequences=1)[0]['generated_text']

print("=== CHAIN-OF-THOUGHT ===\n")
print(f"Prompt:\n{cot_prompt}")
print(f"\nRespuesta:\n{response_cot[len(cot_prompt):].strip()}")

In [ ]:
# Comparación directa: problema sin CoT vs con CoT
problem = "P: Si una camisa cuesta 25 euros y tiene un descuento del 20%, ¿cuál es el precio final?"

# Sin CoT
response_no_cot = generator(problem, max_length=50)[0]['generated_text']

# Con CoT
cot_problem = problem + "\nPensémoslo paso a paso:"
response_cot = generator(cot_problem, max_length=100)[0]['generated_text']

print("=== COMPARACIÓN SIN COT VS CON COT ===")
print(f"\nProblema: {problem}")
print(f"\nSin CoT: {response_no_cot[len(problem):].strip()}")
print(f"\nCon CoT: {response_cot[len(cot_problem):].strip()}")

---
## 5. Introducción a LoRA (Low-Rank Adaptation)

LoRA permite fine-tuning eficiente congelando el modelo base y añadiendo matrices de bajo rango entrenables.

In [ ]:
# Configuración de LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,  # rango
    lora_alpha=32,
    target_modules=["c_attn"],  # para GPT-2, los módulos de atención
    lora_dropout=0.05,
    bias="none"
)

# Aplicar LoRA al modelo
lora_model = get_peft_model(model, lora_config)

# Ver parámetros entrenables
lora_model.print_trainable_parameters()

# Comparación de tamaño
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

total_base = count_parameters(model)
trainable_lora = count_parameters(lora_model) - total_base

print(f"\nParámetros totales modelo base: {total_base:,}")
print(f"Parámetros adicionales LoRA: {trainable_lora:,}")
print(f"Reducción: {trainable_lora/total_base*100:.4f}% del modelo base")

### 5.1. Simulación de Fine-Tuning con LoRA

Creamos un dataset pequeño para fine-tuning (simulado).

In [ ]:
# Dataset de ejemplo para fine-tuning (instrucciones)
train_data = [
    {"instruction": "Traduce 'hello' al español", "output": "hola"},
    {"instruction": "Traduce 'good morning' al español", "output": "buenos días"},
    {"instruction": "Traduce 'thank you' al español", "output": "gracias"},
    {"instruction": "Traduce 'goodbye' al español", "output": "adiós"},
]

# Preparar prompts para fine-tuning
def format_instruction(example):
    return f"Instrucción: {example['instruction']}\nRespuesta: {example['output']}{tokenizer.eos_token}"

train_texts = [format_instruction(ex) for ex in train_data]

print("Ejemplo de entrenamiento:")
print(train_texts[0])

In [ ]:
# Configurar optimizador (solo parámetros LoRA)
optimizer = torch.optim.AdamW(lora_model.parameters(), lr=5e-4)

# Simulación de un paso de entrenamiento
lora_model.train()
for epoch in range(3):
    for text in train_texts:
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(device)
        outputs = lora_model(**inputs, labels=inputs['input_ids'])
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

print("\nFine-tuning con LoRA completado (simulado).")

In [ ]:
# Probar el modelo fine-tuneado
lora_model.eval()
test_prompt = "Instrucción: Traduce 'hello' al español\nRespuesta:"
inputs = tokenizer(test_prompt, return_tensors='pt').to(device)
with torch.no_grad():
    outputs = lora_model.generate(**inputs, max_new_tokens=20)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Prompt: {test_prompt}")
print(f"Respuesta: {response[len(test_prompt):].strip()}")

---
## 6. Cuantización de Modelos

La cuantización reduce la precisión de los pesos para disminuir el tamaño del modelo y acelerar la inferencia.

In [ ]:
# Función para medir tamaño del modelo
def get_model_size(model):
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    size_mb = (param_size + buffer_size) / 1024**2
    return size_mb

# Tamaño original (FP32)
original_size = get_model_size(model)
print(f"Tamaño original del modelo (FP32): {original_size:.2f} MB")

### 6.1. Cuantización Dinámica (INT8)

PyTorch ofrece cuantización dinámica para modelos de lenguaje, que cuantiza los pesos a INT8 durante la inferencia.

In [ ]:
# Aplicar cuantización dinámica
quantized_model = torch.ao.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)

# Tamaño cuantizado
quantized_size = get_model_size(quantized_model)
print(f"Tamaño cuantizado (INT8): {quantized_size:.2f} MB")
print(f"Reducción: {(1 - quantized_size/original_size)*100:.1f}%")

### 6.2. Comparación de Velocidad de Inferencia

Medimos el tiempo de inferencia antes y después de la cuantización.

In [ ]:
def measure_inference_time(model, tokenizer, prompt, n_runs=10):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    model.eval()
    
    # Warmup
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=20)
    
    torch.cuda.synchronize()
    times = []
    for _ in range(n_runs):
        start = time.time()
        with torch.no_grad():
            _ = model.generate(**inputs, max_new_tokens=20)
        torch.cuda.synchronize()
        times.append(time.time() - start)
    
    return np.mean(times), np.std(times)

prompt_test = "Traduce 'hello' al español:"

# Medir tiempo modelo original
mean_orig, std_orig = measure_inference_time(model, tokenizer, prompt_test)
print(f"Modelo original - Tiempo medio: {mean_orig:.3f}s ± {std_orig:.3f}s")

# Mover modelo cuantizado al dispositivo
quantized_model.to(device)
mean_quant, std_quant = measure_inference_time(quantized_model, tokenizer, prompt_test)
print(f"Modelo cuantizado - Tiempo medio: {mean_quant:.3f}s ± {std_quant:.3f}s")

In [ ]:
# Comparación cualitativa de respuestas
def compare_responses(model1, model2, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        out1 = model1.generate(**inputs, max_new_tokens=20)
        out2 = model2.generate(**inputs, max_new_tokens=20)
    
    resp1 = tokenizer.decode(out1[0], skip_special_tokens=True)[len(prompt):].strip()
    resp2 = tokenizer.decode(out2[0], skip_special_tokens=True)[len(prompt):].strip()
    
    return resp1, resp2

prompt_comp = "¿Cuál es la capital de Francia?"
resp_orig, resp_quant = compare_responses(model, quantized_model, tokenizer, prompt_comp)

print(f"Prompt: {prompt_comp}")
print(f"Modelo original: {resp_orig}")
print(f"Modelo cuantizado: {resp_quant}")

---
## 7. (Opcional) Exportación a ONNX

ONNX permite exportar modelos para inferencia en diferentes plataformas.

In [ ]:
# Instalar onnx y onnxruntime
!pip install onnx onnxruntime

# Exportar modelo a ONNX (simplificado)
dummy_input = tokenizer("hello", return_tensors='pt')['input_ids'].to(device)

try:
    torch.onnx.export(
        model,
        dummy_input,
        "model.onnx",
        input_names=['input_ids'],
        output_names=['logits'],
        dynamic_axes={'input_ids': {0: 'batch_size', 1: 'sequence'}}
    )
    print("Modelo exportado a ONNX correctamente.")
except Exception as e:
    print(f"Error al exportar: {e}")
    print("Nota: La exportación a ONNX puede ser compleja para modelos generativos.")

---
## 8. Ejercicio para el Estudiante

1. **Crea tus propios prompts**: Diseña prompts zero-shot, few-shot y CoT para una tarea específica (ej. generación de poemas, resolución de acertijos).

2. **Experimenta con diferentes valores de r en LoRA**: Prueba r=4, r=16 y observa cómo cambia el número de parámetros entrenables.

3. **Compara diferentes niveles de cuantización**: Si tienes acceso, prueba INT8 vs FP16.

4. **Mide la perplejidad**: Antes y después de la cuantización para evaluar la pérdida de calidad.

In [ ]:
# Espacio para el estudiante
# ...

---
## 9. Conclusiones

En este notebook hemos explorado técnicas avanzadas para trabajar con LLMs:

✔️ **Prompt Engineering**:
- Zero-shot funciona para tareas simples.
- Few-shot mejora la precisión con ejemplos.
- Chain-of-Thought es esencial para razonamiento complejo.

✔️ **LoRA**:
- Reduce drásticamente los parámetros entrenables (ej. 0.1% del modelo base).
- Permite fine-tuning eficiente con pocos recursos.

✔️ **Cuantización**:
- Reduce el tamaño del modelo hasta un 75%.
- Acelera la inferencia con pérdida mínima de calidad.

✔️ **ONNX**:
- Permite portabilidad a diferentes entornos.

**Lección clave**: Combinar prompting inteligente con optimizaciones como LoRA y cuantización permite desplegar modelos potentes incluso en entornos con recursos limitados.

---
**Fin del Notebook Conceptual - Semana 13 (NLP)**